## NBA Player Minutes — NGBoost

**- Owner:** Alex

**- Status:** Exploration only — not a production pipeline yet

### Question

**Does a Normal NGBoost minutes model match quantile XGBoost on the same date walk-forward and the 2025-26 holdout?**

### Decision Criteria

**- Model:** NGBoost `NGBRegressor`, Normal distribution, log score. One fit. Quantiles q05, q10, q50, q90, and q95 are read off that distribution.

**- Primary metric:** q50 pinball (minutes). Lower is better.

**- Secondary metric:** 80% interval (q10–q90) and 90% interval (q05–q95) coverage near their targets.

**- Same rows as** `notebooks/nba/minutes/minutes.ipynb`. Train pool excludes 2025-26. Walk-forward uses a 50% date window stepped forward 10%. The holdout is scored once.

**- Params:** a starting point, not an Optuna lock. Do not retune on the holdout.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

def _repo_root() -> Path:
    starts = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        starts.append(Path(nb).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "pyproject.toml").is_file():
                return candidate
    raise RuntimeError("could not find repo root (pyproject.toml)")

ROOT = _repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.minutes import add_minutes_features


In [2]:
# ── Prop-specific config (NBA MIN, NGBoost) ──────────────────────────────────

HOLDOUT_SEASON = "2025-26"
ID_COLS = ["game_id", "player_id", "season_year", "player_name", "game_date"]
TARGET_COL = "minutes"
ROLE_COL = "starting"
SEED = 42

# QUANTILES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
QUANTILES = [0.05, 0.10, 0.50, 0.90, 0.95]

# Normal + log score is the NGBRegressor default. Depth 3 is NGBoost's default tree.
NGB_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.01,
    minibatch_frac=1.0,
    col_sample=1.0,
    verbose=False,
    verbose_eval=0,
    random_state=SEED,
    early_stopping_rounds=50,
    Base=DecisionTreeRegressor(max_depth=3),
)

MIN_TIERS = {
    "<15 min": lambda a: a < 15,
    "15-24 min": lambda a: (a >= 15) & (a < 24),
    "24-31 min": lambda a: (a >= 24) & (a < 31),
    "31+ min": lambda a: a >= 31,
}

ARTIFACT_STEM = "min_nba_ngboost"

from models.shared.artifacts import load_model_bundle, save_model_bundle
from models.shared.metrics import interval_coverage, pinball_loss
from models.shared.ngboost import evaluate_holdout, run_timeseries_cv, run_walk_forward
from models.shared.splits import prepare_splits


### Data


In [3]:
s20 = pd.read_parquet(ROOT / "data/silver/nba/2019-20/regular_season/player_gamelogs.parquet")
s21 = pd.read_parquet(ROOT / "data/silver/nba/2020-21/regular_season/player_gamelogs.parquet")
s22 = pd.read_parquet(ROOT / "data/silver/nba/2021-22/regular_season/player_gamelogs.parquet")
s23 = pd.read_parquet(ROOT / "data/silver/nba/2022-23/regular_season/player_gamelogs.parquet")
s24 = pd.read_parquet(ROOT / "data/silver/nba/2023-24/regular_season/player_gamelogs.parquet")
s25 = pd.read_parquet(ROOT / "data/silver/nba/2024-25/regular_season/player_gamelogs.parquet")
s26 = pd.read_parquet(ROOT / "data/silver/nba/2025-26/regular_season/player_gamelogs.parquet")
df = pd.concat([s20, s21, s22, s23, s24, s25, s26])
df["starting"] = (
    df["start_position"].astype("string").str.strip().isin(["G", "F", "C"]).astype(int)
)
df.sample(5)


,season_year,player_id,player_name,nickname,team_id,team_abbreviation,team_name,game_id,game_date,matchup,...,opp_pace,opp_pace_per40,opp_poss,opp_pie,season_type,pos,game_total,team_spread,player_team_spread,starting
25071,2024-25,1629008,Michael Porter Jr.,Michael,1610612743,DEN,Denver Nuggets,0022400113,2024-10-29T00:00:00,DEN @ BKN,...,95.55,79.62,106,0.474,Regular Season,SF,218.5,5.5,-5.5,1
3970,2022-23,1628975,Jevon Carter,Jevon,1610612749,MIL,Milwaukee Bucks,0022201045,2023-03-16T00:00:00,MIL vs. IND,...,106.00,88.33,106,0.545,Regular Season,PG,236.0,-12.0,-12.0,0
23045,2021-22,1630256,Jae'Sean Tate,Jae'Sean,1610612745,HOU,Houston Rockets,0022100133,2021-11-06T00:00:00,HOU @ DEN,...,97.50,81.25,97,0.479,Regular Season,PF,215.0,-11.5,11.5,1
1437,2022-23,201609,Goran Dragic,Goran,1610612749,MIL,Milwaukee Bucks,0022201174,2023-04-02T00:00:00,MIL vs. PHI,...,92.50,77.08,92,0.395,Regular Season,PG,236.0,-5.0,-5.0,0
21596,2025-26,1629652,Luguentz Dort,Luguentz,1610612760,OKC,Oklahoma City Thunder,0022500055,2025-11-21T00:00:00,OKC @ UTA,...,104.00,86.67,104,0.366,Regular Season,SF,234.0,16.5,-16.5,1


In [4]:
df = add_minutes_features(df)


In [5]:
# Starter availability. Same construction as minutes.ipynb so MIN_FEATURES match.
N = 10  # lookback window in team games

d = df.copy()
d["game_date"] = pd.to_datetime(d["game_date"])
d["mins"] = d["minutes"]
d["started"] = d["starting"].fillna(0).astype(int)
d["played"] = (d["mins"] > 0).astype(int)

sched = (
    d[["season_year", "team_id", "game_id", "game_date"]]
    .drop_duplicates(["team_id", "game_id"])
    .sort_values(["team_id", "game_date"])
)
sched["tg_num"] = sched.groupby(["season_year", "team_id"]).cumcount()

tenure = (
    d.groupby(["season_year", "team_id", "player_id"])["game_date"]
    .agg(first="min", last="max")
    .reset_index()
)
grid = tenure.merge(sched, on=["season_year", "team_id"])
grid = grid[(grid["game_date"] >= grid["first"]) & (grid["game_date"] <= grid["last"])]
grid = grid.merge(
    d[["team_id", "game_id", "player_id", "mins", "started", "played"]],
    on=["team_id", "game_id", "player_id"],
    how="left",
)
grid[["mins", "started", "played"]] = grid[["mins", "started", "played"]].fillna(0)
grid = grid.sort_values(["season_year", "team_id", "player_id", "tg_num"])

g = grid.groupby(["season_year", "team_id", "player_id"])
grid["starts_lN"] = g["started"].transform(
    lambda s: s.shift(1).rolling(N, min_periods=1).sum()
)
grid["mins_played"] = grid["mins"].where(grid["played"] == 1)
grid["min_avg_lN"] = g["mins_played"].transform(
    lambda s: s.shift(1).rolling(N, min_periods=1).mean()
)

grid = grid.sort_values(
    ["team_id", "game_id", "starts_lN", "min_avg_lN"],
    ascending=[True, True, False, False],
)
grid["starter_rank"] = grid.groupby(["team_id", "game_id"]).cumcount() + 1
grid["is_main_starter"] = (
    (grid["starter_rank"] <= 5) & (grid["starts_lN"] > 0)
).astype(int)

main = grid[grid["is_main_starter"] == 1]
team_feats = (
    main.assign(missing_min=main["min_avg_lN"].where(main["played"] == 0, 0))
    .groupby(["team_id", "game_id"])
    .agg(
        main_starters_playing=("played", "sum"),
        main_starters_defined=("player_id", "size"),
        starter_min_missing=("missing_min", "sum"),
    )
    .reset_index()
)

df = df.merge(team_feats, on=["team_id", "game_id"], how="left").merge(
    grid[
        [
            "team_id",
            "game_id",
            "player_id",
            "is_main_starter",
            "min_avg_lN",
            "starter_rank",
        ]
    ],
    on=["team_id", "game_id", "player_id"],
    how="left",
)

own_missing = np.where(
    (df["is_main_starter"] == 1) & (df["minutes"].fillna(0) == 0),
    df["min_avg_lN"].fillna(0),
    0,
)
df["starter_min_missing_teammates"] = df["starter_min_missing"] - own_missing
df["main_starters_out"] = df["main_starters_defined"] - df["main_starters_playing"]


In [6]:
MIN_FEATURES = [
    "starting",
    "min_ewm_hl_3",
    "min_lag_1",
    "season_min_mean",
    "current_team_min_mean",
    "min10_x_expected_possessions",
    "season_start_rate",
    "start_rate_10",
    "player_days_since_appearance",
    "player_appearances_prev_7d",
    "prior_season_min_mean",
    "start_rate_x_abs_spread",
    "implied_team_total",
    "min_mean_3_minus_10",
    "starter_min_missing_teammates",
    "main_starters_out",
    "min_avg_lN",
    "starter_rank",
    "is_main_starter",
    "fouls_per_36_10",
]

print(f"Number of features: {len(MIN_FEATURES)}")
print(f"Number of nulls: {df[MIN_FEATURES].isna().sum().sum()}")


Number of features: 20
Number of nulls: 85602


### Train / validation / test

**- Train:** seasons through 2024-25. Each walk-forward fold fits on its past dates. The 2025-26 fit uses the first 90% of this pool.

**- Validation:** inside the train pool only. Four non-overlapping blocks, each the next 10% of game dates after a 50% training window. Early stopping uses the last 10% of that fold's training dates. The validation block is only scored.

**- Test:** 2025-26. Scored once. Those rows are never in an early-stopping set.


In [7]:
df = df[df["minutes"] > 0].sort_values("game_date", kind="mergesort")
print(f"After filtering minutes: {len(df):,} rows")

splits = prepare_splits(
    df,
    holdout_season=HOLDOUT_SEASON,
    features=MIN_FEATURES,
    target_col=TARGET_COL,
    id_cols=ID_COLS,
    role_col=ROLE_COL,
    extra_keep=["minutes"],
)
ppm_df = splits["ppm_df"]
ppm_holdout = splits["ppm_holdout"]
X = splits["X"]
y = splits["y"]
print(f"Train pool (excl. {HOLDOUT_SEASON}) : {len(ppm_df):,} rows")
print(f"Holdout   ({HOLDOUT_SEASON})        : {len(ppm_holdout):,} rows")
print(f"Date range (train)   : {ppm_df['game_date'].min().date()} → {ppm_df['game_date'].max().date()}")
print(f"Date range (holdout) : {ppm_holdout['game_date'].min().date()} → {ppm_holdout['game_date'].max().date()}")


After filtering minutes: 176,710 rows
Train pool (excl. 2025-26) : 150,062 rows
Holdout   (2025-26)        : 26,648 rows
Date range (train)   : 2019-10-22 → 2025-04-13
Date range (holdout) : 2025-10-21 → 2026-04-12
Train pool (excl. 2025-26) : 150,062 rows
Holdout   (2025-26)        : 26,648 rows
Date range (train)   : 2019-10-22 → 2025-04-13
Date range (holdout) : 2025-10-21 → 2026-04-12


### Walk-forward


In [8]:
tscv_results = run_timeseries_cv(
    X,
    y,
    ppm_df,
    ngb_params=NGB_PARAMS,
    role_col=ROLE_COL,
    tiers=MIN_TIERS,
    quantiles=QUANTILES,
)
wf = run_walk_forward(
    X,
    y,
    ppm_df,
    ngb_params=NGB_PARAMS,
    role_col=ROLE_COL,
    tiers=MIN_TIERS,
    quantiles=QUANTILES,
)
wf_results = wf["wf_results"]


── Phase 1: TimeSeriesSplit (NGBoost) ──────────────────────────

TSCV fold 1
  n=25010
  pinball  | q0.05=0.619  q0.10=1.020  q0.50=2.252  q0.90=0.992  q0.95=0.592
  coverage | 80%=80.9% (target 80%)  90%=90.4% (target 90%)
  Starters   | n=11707 | pinball q50: 2.131 | 80% coverage: 81.5%
  Bench      | n=13303 | pinball q50: 2.359 | 80% coverage: 80.4%
  minute tiers by predicted q50
  <15 min    | n= 5602 | pinball q50: 2.451 | 80% coverage: 81.5%
  15-24 min  | n= 7486 | pinball q50: 2.374 | 80% coverage: 78.3%
  24-31 min  | n= 6554 | pinball q50: 2.227 | 80% coverage: 81.2%
  31+ min    | n= 5368 | pinball q50: 1.908 | 80% coverage: 83.8%

TSCV fold 2
  n=25010
  pinball  | q0.05=0.653  q0.10=1.068  q0.50=2.335  q0.90=1.023  q0.95=0.611
  coverage | 80%=80.5% (target 80%)  90%=89.8% (target 90%)
  Starters   | n=11824 | pinball q50: 2.202 | 80% coverage: 81.2%
  Bench      | n=13186 | pinball q50: 2.453 | 80% coverage: 79.8%
  minute tiers by predicted q50
  <15 min    | n= 5504 

### Holdout


In [11]:
holdout = evaluate_holdout(
    ppm_df,
    ppm_holdout,
    features=MIN_FEATURES,
    target_col=TARGET_COL,
    ngb_params=NGB_PARAMS,
    role_col=ROLE_COL,
    tiers=MIN_TIERS,
    wf_results=wf_results,
    quantiles=QUANTILES,
    fold_label=f"{HOLDOUT_SEASON} Blind Holdout",
)
models_ho = holdout["models_ho"]
preds_ho = holdout["preds_ho"]
ho_metrics = holdout["ho_metrics"]
X_ho = holdout["X_ho"]
y_ho = holdout["y_ho"]

print(
    f"{HOLDOUT_SEASON} holdout pinball q50: "
    f"{pinball_loss(y_ho, preds_ho['q_0.50'], 0.50):.3f}"
)
print(
    f"{HOLDOUT_SEASON} holdout 80% coverage: "
    f"{interval_coverage(y_ho, preds_ho['q_0.10'], preds_ho['q_0.90']):.1%}"
)


── 2025-26 Blind Holdout ─────────────────────────────────────────────────
  Train pool: 90% rows for fit, remainder for early-stopping only; predict holdout (never in eval_set).

2025-26 Blind Holdout
  n=26648
  pinball  | q0.05=0.633  q0.10=1.032  q0.50=2.251  q0.90=1.017  q0.95=0.607
  coverage | 80%=81.9% (target 80%)  90%=90.8% (target 90%)
  Starters   | n=12300 | pinball q50: 2.095 | 80% coverage: 83.5%
  Bench      | n=14348 | pinball q50: 2.385 | 80% coverage: 80.6%
  minute tiers by predicted q50
  <15 min    | n= 6292 | pinball q50: 2.519 | 80% coverage: 81.0%
  15-24 min  | n= 7542 | pinball q50: 2.334 | 80% coverage: 79.8%
  24-31 min  | n= 7935 | pinball q50: 2.183 | 80% coverage: 82.5%
  31+ min    | n= 4879 | pinball q50: 1.889 | 80% coverage: 85.6%

───────────────────────────────────────────────────────
  Walk-forward pinball q50 (mean) : 2.268
  Holdout pinball q50             : 2.251
  Pinball gap                     : -0.017  ✓ acceptable
2025-26 holdout pinball q

### Save


In [10]:
metadata = {
    "model": "ngboost",
    "distribution": "Normal",
    "quantiles": list(QUANTILES),
    "holdout_season": HOLDOUT_SEASON,
    "ngb_params": {
        key: NGB_PARAMS[key]
        for key in NGB_PARAMS
        if key != "Base"
    },
    "base_max_depth": NGB_PARAMS["Base"].get_params()["max_depth"],
}

save_path = save_model_bundle(
    models_ho,
    train_df=ppm_df,
    holdout_df=ppm_holdout,
    wf_results=wf_results,
    ho_metrics=ho_metrics,
    features=MIN_FEATURES,
    artifact_stem=ARTIFACT_STEM,
    metadata=metadata,
)
bundle = load_model_bundle(save_path)
loaded = bundle["quantile_models"]
loaded_preds = pd.DataFrame({key: loaded[key].predict(X_ho) for key in preds_ho})
print("Saved quantiles:", ", ".join(loaded_preds.columns))
print(
    f"Loaded model pinball q50 (holdout): "
    f"{pinball_loss(y_ho, loaded_preds['q_0.50'], 0.50):.3f}"
)


Saved to /Users/alexgonzalez/Documents/nba_quant/models/saved_models/min_nba_ngboost_2026-04-12.joblib
Loaded /Users/alexgonzalez/Documents/nba_quant/models/saved_models/min_nba_ngboost_2026-04-12.joblib
Saved quantiles: q_0.05, q_0.10, q_0.50, q_0.90, q_0.95
Loaded model pinball q50 (holdout): 2.251
